# 02 · Train the patent-drawing style LoRA (SDXL, Colab L4)

Runs the Hugging Face Diffusers SDXL LoRA script with `configs/train_sdxl.yaml`.

* **Runtime:** L4 GPU. Expect roughly 1-2 hours and about 2-4 compute units for 1,500 steps.
* **Checkpoints go to Google Drive** every 250 steps. If Colab disconnects, re-run all cells:
  training resumes from the latest checkpoint.
* **When it finishes, disconnect the runtime** (last cell) so it stops using units.
* Keep this tab open while training runs: on Colab Pro, closing it can end the session.

In [ ]:
# --- Check the GPU before spending units ---------------------------------
# Runtime -> Change runtime type -> L4. An A100 works but burns units several times faster.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Mount Drive and open the project (lives in MyDrive/patent-lora/repo) ----
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess, zipfile
DRIVE_ROOT = "/content/drive/MyDrive/patent-lora"   # project, checkpoints and results all live here
REPO_DIR   = f"{DRIVE_ROOT}/repo"
ZIP_PATH   = f"{DRIVE_ROOT}/patent-drawing-lora.zip"  # upload the project zip here once
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError("Upload patent-drawing-lora.zip to MyDrive/patent-lora/ in Google Drive, then re-run.")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DRIVE_ROOT)
    shutil.move(f"{DRIVE_ROOT}/patent-drawing-lora", REPO_DIR)
    print("Project unpacked to", REPO_DIR)
%cd {REPO_DIR}

In [ ]:
# --- Pinned installs (about 2-3 minutes) ----------------------------------
# The Diffusers training script requires Diffusers from source, so both are pinned to one commit.
DIFFUSERS_COMMIT = "cda991a69ad21b8d3239a307d58538fa1a3a6b00"
!pip -q install "git+https://github.com/huggingface/diffusers@{DIFFUSERS_COMMIT}"
!pip -q install "accelerate>=0.22.0" "transformers>=4.25.1" "peft>=0.17.0" datasets ftfy tensorboard Jinja2 bitsandbytes pyyaml
!wget -q -O /content/train_lora_sdxl.py https://raw.githubusercontent.com/huggingface/diffusers/{DIFFUSERS_COMMIT}/examples/text_to_image/train_text_to_image_lora_sdxl.py
!accelerate config default --mixed_precision bf16

## Check the dataset
Every image needs a caption line in `dataset/images/metadata.jsonl` (built by notebook 01).

In [ ]:
import json, glob
images = sorted(glob.glob("dataset/images/*.png"))
captions = [json.loads(l) for l in open("dataset/images/metadata.jsonl")]
print(f"{len(images)} images, {len(captions)} captions")
assert len(images) == len(captions) and len(images) > 0, "image/caption count mismatch"
for c in captions[:5]:
    print(" ", c["text"])
assert all(c["text"].startswith("ptnt_style, ") for c in captions), "every caption must start with the trigger"

## Train
Flags come from the YAML so every run is recorded. The config is copied next to the checkpoints.

In [ ]:
import yaml, shutil, shlex
cfg = yaml.safe_load(open("configs/train_sdxl.yaml"))
run_name = cfg.pop("run_name")
OUTPUT_DIR = f"{DRIVE_ROOT}/output/{run_name}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
shutil.copy("configs/train_sdxl.yaml", f"{OUTPUT_DIR}/train_sdxl.yaml")

flags = []
for k, v in cfg.items():
    if v is True:
        flags.append(f"--{k}")
    elif v is not False and v is not None:
        flags += [f"--{k}", str(v)]
flags += ["--output_dir", OUTPUT_DIR, "--resume_from_checkpoint", "latest"]
cmd = "accelerate launch /content/train_lora_sdxl.py " + " ".join(shlex.quote(f) for f in flags)
print(cmd)

In [ ]:
!{cmd}

## Pick the checkpoint by eye

Renders the same 4 prompts at a fixed seed from every saved checkpoint.
Pick the **earliest** checkpoint where backgrounds are pure white, lines are crisp and the object is still right.

In [ ]:
import sys, glob, torch
sys.path.insert(0, "scripts")
from eval import load_pipe, make_grid
from pixel_metrics import all_metrics

ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
if os.path.exists(f"{OUTPUT_DIR}/pytorch_lora_weights.safetensors"):
    ckpts.append(OUTPUT_DIR)  # final weights are saved in the output dir itself when training completes
probe = [("a quadcopter drone", "perspective view"), ("a kettle", "side view"),
         ("a smartwatch", "front view"), ("a desk lamp", "perspective view")]
NEG = "color, gray background, photo, photorealistic, 3D render, blurry, text, watermark"

pipe = load_pipe(None, offload=False)
cells, cols = {}, []
for ck in ckpts:
    name = "final" if ck == OUTPUT_DIR else ck.rsplit("-", 1)[1]
    pipe.load_lora_weights(ck, weight_name="pytorch_lora_weights.safetensors", adapter_name=f"c{name}")
    pipe.set_adapters([f"c{name}"], adapter_weights=[0.8])
    for i, (obj, view) in enumerate(probe):
        g = torch.Generator("cpu").manual_seed(42)
        im = pipe(f"ptnt_style, {view} of {obj}", negative_prompt=NEG, num_inference_steps=30,
                  guidance_scale=6.0, generator=g).images[0]
        cells[(i, f"step {name}")] = im
        print(name, obj, {k: round(v, 3) for k, v in all_metrics(im).items()})
    cols.append(f"step {name}")
    pipe.delete_adapters([f"c{name}"])

grid = make_grid(cells, [o for o, _ in probe], cols)
grid.save(f"{OUTPUT_DIR}/checkpoint_comparison.png")
grid

## Disconnect
Stops the runtime so it no longer uses compute units. Your checkpoints are already on Drive.

In [ ]:
from google.colab import runtime
runtime.unassign()